# Notebook 4: Carga a Base de Datos (Gold + SQLite)
**Autores:** Leonardo Figueroa, Miguel Flores, Steven Villegas.

**Pipeline ETL - NYC Taxi Trip Records**

**Objetivo:** Construir la capa Gold con agregaciones analíticas y cargar los resultados en SQLite.


## Configuración Inicial


In [1]:
import sys, os, json, yaml, logging
from datetime import datetime
from pathlib import Path

# --- MEMORIA ---
os.environ['PYSPARK_SUBMIT_ARGS'] = '--driver-memory 4g --executor-memory 4g pyspark-shell'
# ---------------

sys.path.append(os.path.abspath('..'))
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger('etl')
from pyspark.sql import SparkSession, functions as F

_config_path = os.path.join(os.path.abspath('..'), "config", "etl_config.yaml")
with open(_config_path, "r", encoding="utf-8") as _f:
    _cfg = yaml.safe_load(_f)
_spark_cfg = _cfg.get("spark", {})
builder = SparkSession.builder \
    .appName("NYC_Taxi_ETL_04") \
    .master(_spark_cfg.get("master", "local[*]"))
for k, v in _spark_cfg.get("config", {}).items():
    builder = builder.config(k, v)
spark = builder.getOrCreate()

from src.config_loader import *
from src.load import build_gold, load_to_sqlite, run_verification_queries


---
## FASE 6: Enriquecimiento Analítico (Gold)
**Objetivo:** Construir la capa Gold con 3 datasets agregados listos para consultas analíticas.


In [2]:
result = build_gold(spark, SERVICES)
if result:
    df_gold_trips_clean, df_gold_daily_revenue, df_gold_location_performance = result
    print(f"\ngold_trips_clean: {df_gold_trips_clean.count()} registros")
    print(f"gold_daily_revenue: {df_gold_daily_revenue.count()} registros")
    print(f"gold_location_performance: {df_gold_location_performance.count()} registros")
else:
    logger.warning("No gold data built - silver data may be empty")
    raise SystemExit("Stopping: no gold data")


2026-06-17 22:57:16,028 - src.load - INFO - Union all silver: 673147407 total records
2026-06-17 23:09:23,059 - src.load - INFO - gold_trips_clean: 665887486 records
2026-06-17 23:10:23,781 - src.load - INFO - gold_daily_revenue: 2564 records
2026-06-17 23:11:42,810 - src.load - INFO - gold_location_performance: 149335 records



gold_trips_clean: 665887486 registros
gold_daily_revenue: 2564 registros
gold_location_performance: 149335 registros


In [3]:
print("\nMuestra de gold_trips_clean:")
df_gold_trips_clean.show(5, truncate=False)
print("\nMuestra de gold_daily_revenue:")
df_gold_daily_revenue.show(5, truncate=False)
print("\nMuestra de gold_location_performance:")
df_gold_location_performance.show(5, truncate=False)



Muestra de gold_trips_clean:
+----------------------------------------------------------------+------------+-------------------+-------------------+---------------------+-------------+------------------+-------------------+------------+-----------+----------+------------+------------------+------------------+-----------------+---------------+----+-----+-------------------------------+
|trip_id                                                         |service_type|pickup_datetime    |dropoff_datetime   |trip_duration_minutes|trip_distance|pickup_location_id|dropoff_location_id|payment_type|fare_amount|tip_amount|total_amount|tip_percentage    |average_speed_mph |fare_per_mile    |is_airport_trip|year|month|source_file                    |
+----------------------------------------------------------------+------------+-------------------+-------------------+---------------------+-------------+------------------+-------------------+------------+-----------+----------+------------+---------

---
## FASE 7: Carga en Base de Datos (SQLite)
**Objetivo:** Persistir los resultados en SQLite con 6 tablas para consultas posteriores.


In [2]:
try:
    engine = load_to_sqlite(spark)
    logger.info("SQLite load completed successfully")
except Exception as e:
    logger.error(f"SQLite load failed: {e}")
    engine = None


2026-06-18 00:53:07,045 - src.load - INFO - Loaded gold_daily_revenue: 2564 rows
2026-06-18 00:53:09,098 - src.load - INFO - Loaded gold_location_performance: 149335 rows
2026-06-18 00:53:09,393 - src.load - INFO - Loaded quality_rejected_records: 2593 rows
2026-06-18 00:53:09,645 - src.load - INFO - Loaded quality_metrics_summary: 84 rows
2026-06-18 00:53:09,859 - src.load - INFO - Loaded audit_file_inventory: 84 rows
2026-06-18 00:53:12,731 - src.load - INFO - Loaded chunk 1/567: part-00000-576724a0-a25e-40ca-8102-c6b927d4c886-c000.snappy.parquet (102335 rows)
2026-06-18 00:53:14,478 - src.load - INFO - Loaded chunk 2/567: part-00001-576724a0-a25e-40ca-8102-c6b927d4c886-c000.snappy.parquet (102292 rows)
2026-06-18 00:53:16,325 - src.load - INFO - Loaded chunk 3/567: part-00002-576724a0-a25e-40ca-8102-c6b927d4c886-c000.snappy.parquet (102085 rows)
2026-06-18 00:53:18,335 - src.load - INFO - Loaded chunk 4/567: part-00003-576724a0-a25e-40ca-8102-c6b927d4c886-c000.snappy.parquet (102007

---
## Consultas de Verificación
**Objetivo:** Validar los resultados del pipeline mediante 3 consultas SQL.


In [3]:
if engine is not None:
    run_verification_queries(engine)
else:
    logger.error("Cannot run queries - no engine available")



  Q1 - Revenue por servicio
service_type  total_trips  total_revenue
      yellow      3371576    98928891.74

  Q2 - Metricas de calidad
service_type  year  month  total_records  valid_records  rejected_records  quality_percentage
       fhvhv  2024      1       19663879       19657231              6648               99.97
       green  2024      1          56551          53103              3448               93.90
      yellow  2024      1        2964624        2864053            100571               96.61
       fhvhv  2024      2       19359093       19344469             14624               99.92
       green  2024      2          53577          50234              3343               93.76
      yellow  2024      2        3007525        2896203            111322               96.30
       fhvhv  2024      3       21280737       21273247              7490               99.96
       green  2024      3          57457          53913              3544               93.83
      yellow  2

2026-06-18 00:54:29,531 - src.load - INFO - Verification queries completed


 pickup_location_id  dropoff_location_id  total_trips  total_revenue  avg_duration
                132                  265         6463      846401.20     45.196712
                132                  230         6327      609465.64     62.707036
                138                  230         5917      483183.15     42.475593
                237                  236        22974      366989.23      7.609238
                132                  164         3443      335149.68     52.913389
                236                  237        19696      322661.08      8.296905
                132                   48         3117      298271.75     62.018089
                230                  138         3636      290808.40     36.884782
                138                  161         3690      288773.79     39.280994
                132                  161         2842      276173.06     58.211078
                132                  163         2664      252555.29     61.931788
    

---
## Resumen de Carga
Datos cargados exitosamente en SQLite. Las 6 tablas están disponibles para consultas analíticas.
